# ActionRank on Colab

Runtime -> Change runtime type -> **GPU (A100 or L4)**. Then run the cells top to bottom.

1. Upload `actionrank_colab.zip` when prompted (code + processed ToolBench data, ~25 MB).
2. The fine-tuned generation baseline trains (~15-25 min on A100).
3. Optionally Tier 2 at scale (LoRA + span head on all training steps, ~30-40 min).
4. Download `actionrank_results.zip` and unzip it in the repo root on your Mac; then run
   `.venv/bin/python eval.py --systems tier1,span,baseline,baseline_sft` locally.

In [ ]:
from google.colab import files
up = files.upload()  # pick actionrank_colab.zip
!rm -rf ActionRank && mkdir ActionRank && unzip -q actionrank_colab.zip -d ActionRank && ls ActionRank

In [ ]:
%cd /content/ActionRank
!pip -q install -r requirements.txt
import os, torch
os.environ['ACTIONRANK_DEVICE'] = 'cuda'
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# bigger batches on a real GPU; same effective batch (16) as the local runs
!sed -i 's/  sft_batch_size: 1/  sft_batch_size: 8/; s/  sft_grad_accum: 16/  sft_grad_accum: 2/' config.yaml
!grep -nE 'sft_batch_size|sft_grad_accum' config.yaml

In [ ]:
# 1) fine-tuned generation baseline (all 10,568 training steps, 1 epoch)
!ACTIONRANK_DEVICE=cuda python baseline_sft.py 2>&1 | grep -v 'HTTP Request'

In [ ]:
# 2) optional: Tier 2 at scale -- LoRA + table head on all training steps, 2 epochs
RUN_TIER2 = True
if RUN_TIER2:
    !sed -i 's/  max_train_examples: 400/  max_train_examples: 10568/; s/^  epochs: 1$/  epochs: 2/; s/  batch_size: 2$/  batch_size: 8/; s/  grad_accum: 8/  grad_accum: 2/' config.yaml
    !grep -nA9 '^tier2:' config.yaml
    !ACTIONRANK_DEVICE=cuda python train_tier2.py 2>&1 | grep -v 'HTTP Request'

In [ ]:
# 3) package everything to bring back: adapters, heads, histories
!rm -f actionrank_results.zip && zip -qr actionrank_results.zip checkpoints/baseline_sft checkpoints/tier2 results/*.json 2>/dev/null; ls -la actionrank_results.zip
from google.colab import files
files.download('actionrank_results.zip')